# Module 34 — Multi-Armed Bandit: Epsilon-Greedy for News Headlines

> **Track 3 · Yandex Big Tech & Search**  
> **Difficulty**: Intermediate  
> **Runtime**: ~5 minutes  
> **Key Libraries**: Pandas, NumPy, Plotly  
> **Prerequisite**: Module 6 (Statistical Tests)

## Table of Contents

1. [Business Context](#1-business-context)
2. [Setup & Imports](#2-setup--imports)
3. [Synthetic News Headlines](#3-synthetic-news-headlines)
4. [Epsilon-Greedy Algorithm](#4-epsilon-greedy-algorithm)
5. [UCB Algorithm](#5-ucb-algorithm)
6. [Thompson Sampling](#6-thompson-sampling)
7. [Visualization](#7-visualization)
8. [Key Business Takeaway](#8-key-business-takeaway)

---
## §1 · Business Context

### Why multi-armed bandits for news?

Multi-armed bandits **balance exploration and exploitation**:
- **Exploration**: Try different headlines to learn performance.
- **Exploitation**: Show best-performing headlines more often.
- **Online learning**: Update strategy in real-time.

**Applications at Yandex**:
1. **News headlines**: Test different headline versions.
2. **Ad creative**: Optimize ad copy and images.
3. **Content recommendations**: Explore new content types.

**Key algorithms**:
- **Epsilon-Greedy**: Random exploration with probability ε.
- **UCB**: Upper Confidence Bound for optimistic exploration.
- **Thompson Sampling**: Bayesian approach with posterior sampling.

In this notebook we will:
1. Generate synthetic news headline click data.
2. Implement Epsilon-Greedy, UCB, and Thompson Sampling.
3. Compare cumulative regret across algorithms.
4. Visualize exploration-exploitation tradeoff.

---
## §2 · Setup & Imports

In [ ]:
# ── Reproducibility ──────────────────────────────────────────────
import numpy as np
import pandas as pd
import warnings

np.random.seed(42)
warnings.filterwarnings("ignore")

# ── Visualization ────────────────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px

# ── Display settings ─────────────────────────────────────────────
pd.set_option("display.max_columns", 30)
sns.set_theme(style="whitegrid")

print("✓ All imports loaded")

---
## §3 · Synthetic News Headlines

In [ ]:
# Generate synthetic news headline click data
N_HEADLINES = 5
N_ROUNDS = 10000

# Headlines with true click rates
headlines = [
    {'id': 0, 'title': 'Breaking: Major Tech Acquisition Announced', 'true_ctr': 0.08},
    {'id': 1, 'title': 'New Study Reveals Surprising Health Benefits', 'true_ctr': 0.12},
    {'id': 2, 'title': 'Stock Market Hits Record High', 'true_ctr': 0.06},
    {'id': 3, 'title': 'Celebrity Couple Announces Engagement', 'true_ctr': 0.15},
    {'id': 4, 'title': 'Local Team Wins Championship', 'true_ctr': 0.10},
]

# True CTRs
true_ctrs = [h['true_ctr'] for h in headlines]
best_headline = np.argmax(true_ctrs)

print(f"✓ Generated {N_HEADLINES} news headlines")
print(f"\nHeadlines and true CTRs:")
for h in headlines:
    print(f"  {h['id']}: {h['title'][:50]}... (CTR: {h['true_ctr']*100:.1f}%)")
print(f"\nBest headline: {best_headline} (CTR: {true_ctrs[best_headline]*100:.1f}%)")

---
## §4 · Epsilon-Greedy Algorithm

In [ ]:
# Implement Epsilon-Greedy
class EpsilonGreedy:
    def __init__(self, n_arms, epsilon=0.1):
        self.n_arms = n_arms
        self.epsilon = epsilon
        self.counts = np.zeros(n_arms)
        self.values = np.zeros(n_arms)
    
    def select_arm(self):
        if np.random.random() < self.epsilon:
            return np.random.randint(self.n_arms)
        return np.argmax(self.values)
    
    def update(self, arm, reward):
        self.counts[arm] += 1
        n = self.counts[arm]
        self.values[arm] = ((n - 1) * self.values[arm] + reward) / n

# Run simulation
def run_bandit(algorithm, n_rounds, true_ctrs):
    rewards = []
    regrets = []
    
    for _ in range(n_rounds):
        arm = algorithm.select_arm()
        reward = np.random.binomial(1, true_ctrs[arm])
        algorithm.update(arm, reward)
        
        rewards.append(reward)
        regret = true_ctrs[best_headline] - true_ctrs[arm]
        regrets.append(regret)
    
    return rewards, regrets

# Run Epsilon-Greedy
eg = EpsilonGreedy(N_HEADLINES, epsilon=0.1)
eg_rewards, eg_regrets = run_bandit(eg, N_ROUNDS, true_ctrs)

print(f"✓ Epsilon-Greedy simulation complete")
print(f"  Total reward: {sum(eg_rewards)}")
print(f"  Cumulative regret: {sum(eg_regrets):.2f}")
print(f"  Average reward: {np.mean(eg_rewards)*100:.2f}%")

---
## §5 · UCB Algorithm

In [ ]:
# Implement UCB (Upper Confidence Bound)
class UCB:
    def __init__(self, n_arms):
        self.n_arms = n_arms
        self.counts = np.zeros(n_arms)
        self.values = np.zeros(n_arms)
        self.total_counts = 0
    
    def select_arm(self):
        # Try each arm once first
        for arm in range(self.n_arms):
            if self.counts[arm] == 0:
                return arm
        
        # UCB formula
        ucb_values = self.values + np.sqrt(2 * np.log(self.total_counts) / self.counts)
        return np.argmax(ucb_values)
    
    def update(self, arm, reward):
        self.counts[arm] += 1
        self.total_counts += 1
        n = self.counts[arm]
        self.values[arm] = ((n - 1) * self.values[arm] + reward) / n

# Run UCB
ucb = UCB(N_HEADLINES)
ucb_rewards, ucb_regrets = run_bandit(ucb, N_ROUNDS, true_ctrs)

print(f"✓ UCB simulation complete")
print(f"  Total reward: {sum(ucb_rewards)}")
print(f"  Cumulative regret: {sum(ucb_regrets):.2f}")
print(f"  Average reward: {np.mean(ucb_rewards)*100:.2f}%")

---
## §6 · Thompson Sampling

In [ ]:
# Implement Thompson Sampling
class ThompsonSampling:
    def __init__(self, n_arms):
        self.n_arms = n_arms
        self.alpha = np.ones(n_arms)  # Success counts + 1
        self.beta = np.ones(n_arms)   # Failure counts + 1
    
    def select_arm(self):
        # Sample from Beta distribution
        samples = np.random.beta(self.alpha, self.beta)
        return np.argmax(samples)
    
    def update(self, arm, reward):
        if reward == 1:
            self.alpha[arm] += 1
        else:
            self.beta[arm] += 1

# Run Thompson Sampling
ts = ThompsonSampling(N_HEADLINES)
ts_rewards, ts_regrets = run_bandit(ts, N_ROUNDS, true_ctrs)

print(f"✓ Thompson Sampling simulation complete")
print(f"  Total reward: {sum(ts_rewards)}")
print(f"  Cumulative regret: {sum(ts_regrets):.2f}")
print(f"  Average reward: {np.mean(ts_rewards)*100:.2f}%")

---
## §7 · Visualization

In [ ]:
# Compare algorithms
fig = make_subplots(rows=1, cols=2, subplot_titles=['Cumulative Regret', 'Arm Selection Distribution'])

# Cumulative regret
fig.add_trace(go.Scatter(
    x=list(range(N_ROUNDS)),
    y=np.cumsum(eg_regrets),
    name='Epsilon-Greedy',
    mode='lines'
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=list(range(N_ROUNDS)),
    y=np.cumsum(ucb_regrets),
    name='UCB',
    mode='lines'
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=list(range(N_ROUNDS)),
    y=np.cumsum(ts_regrets),
    name='Thompson Sampling',
    mode='lines'
), row=1, col=1)

# Arm selection distribution
arm_counts = [eg.counts, ucb.counts, ts.alpha - 1]
for i, (counts, name) in enumerate(zip(arm_counts, ['Epsilon-Greedy', 'UCB', 'Thompson'])):
    fig.add_trace(go.Bar(
        x=list(range(N_HEADLINES)),
        y=counts,
        name=name,
        opacity=0.7
    ), row=1, col=2)

fig.update_layout(height=400, showlegend=True)
fig.show()

print("💡 Key insights:")
print("   - Thompson Sampling typically has lowest regret")
print("   - UCB explores more initially, then exploits")
print("   - Epsilon-Greedy explores uniformly throughout")

---
## §8 · Key Business Takeaway

> ### 🎯 Action Items for Content & Experimentation Teams
>
> | Aspect | Recommendation |
> |--------|---------------|
> | **When to use** | A/B testing, content optimization, ad creative testing |
> | **Algorithm** | Thompson Sampling for best performance |
> | **Arms** | 3-10 variants (headlines, images, etc.) |
> | **Evaluation** | Cumulative regret, convergence speed |
> | **Deployment** | Real-time updates, gradual rollout |
>
> ### 💡 Yandex-Specific Guidelines
>
> 1. **News headline optimization**:
>    ```
>    Show headline → Track clicks → Update model → Repeat
>    ```
>
> 2. **Algorithm comparison**:
>    | Algorithm | Exploration | Regret | Use Case |
>    |-----------|-------------|--------|----------|
>    | Epsilon-Greedy | Random | Medium | Simple setups |
>    | UCB | Optimistic | Low-Medium | Known rewards |
>    | Thompson | Bayesian | Low | Most scenarios |
>
> 3. **Production tips**:
>    - Start with high exploration (ε=0.3)
>    - Gradually reduce exploration over time
>    - Monitor for cold-start problems
>
> ### 🔑 Key Takeaways
>
> 1. **Multi-armed bandits optimize online** without offline training.
> 2. **Thompson Sampling** typically performs best.
> 3. **Exploration-exploitation tradeoff** is fundamental.
> 4. **Cumulative regret** measures algorithm performance.
> 5. **Real-time updates** enable continuous optimization.